In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("creditcard.csv")

# Check for null values
print(df.isnull().sum())

# Remove duplicates
df = df.drop_duplicates()

# Normalize 'Amount' and 'Time'
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['norm_amount'] = scaler.fit_transform(df[['Amount']])
df['norm_time'] = scaler.fit_transform(df[['Time']])

# Drop original 'Amount' and 'Time'
df.drop(['Amount', 'Time'], axis=1, inplace=True)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df.drop('Class', axis=1)
y = df['Class']

# Fit a random forest for feature importance
model = RandomForestClassifier()
model.fit(X, y)

# View feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
important_features = importances.sort_values(ascending=False)
print(important_features.head(10))


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} F1 Score: {f1_score(y_test, y_pred):.4f}")


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, scoring='f1', cv=3)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)


In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=6))
])

pipeline.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipeline.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
import joblib

joblib.dump(pipeline, 'fraud_model.pkl')

# Load it back
model = joblib.load('fraud_model.pkl')


In [ ]:
pip install fastapi uvicorn


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()
model = joblib.load("fraud_model.pkl")

class Transaction(BaseModel):
    features: list  # example: [0.1, -1.2, ..., 0.4]

@app.post("/predict")
def predict(data: Transaction):
    prediction = model.predict([data.features])
    return {"fraud": bool(prediction[0])}


In [ ]:
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d "{\"features\": [0.1, -1.2, 0.3, ..., 0.4]}"


In [ ]:
FROM python:3.9
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
